In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# -------------------------------------------------------
# 1. TENSORS & AUTOGRAD
# -------------------------------------------------------

# Creating tensors
x = torch.tensor([[1., 2.], [3., 4.]])
print("Tensor:\n", x)
print("Shape:", x.shape)
print("Dtype:", x.dtype)
print("Device:", x.device)

# Common initializations
zeros   = torch.zeros(3, 4)
ones    = torch.ones(3, 4)
rand    = torch.randn(3, 4)        # N(0,1)
arange  = torch.arange(0, 10, 2)  # like np.arange

# Operations
a = torch.tensor([1., 2., 3.], requires_grad=True)
b = torch.tensor([4., 5., 6.], requires_grad=True)
c = (a * b).sum()   # scalar
c.backward()
print("\nGrad of a:", a.grad)   # dc/da = b
print("Grad of b:", b.grad)   # dc/db = a


In [ ]:
# -------------------------------------------------------
# 2. CUSTOM DATASET
# -------------------------------------------------------
from torch.utils.data import Dataset, DataLoader
import torch

class RegressionDataset(Dataset):
    """Simple dataset: y = 3x + noise"""
    def __init__(self, n_samples=200):
        self.X = torch.randn(n_samples, 1)
        self.y = 3 * self.X + 0.5 * torch.randn(n_samples, 1)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

dataset = RegressionDataset(n_samples=200)
print("Dataset size:", len(dataset))
print("Sample[0]:", dataset[0])


In [ ]:
# -------------------------------------------------------
# 3. DATALOADER — batching, shuffling, splitting
# -------------------------------------------------------
from torch.utils.data import random_split

# Train / val split
train_size = int(0.8 * len(dataset))
val_size   = len(dataset) - train_size
train_ds, val_ds = random_split(dataset, [train_size, val_size])

# Loaders
train_loader = DataLoader(
    train_ds,
    batch_size=32,
    shuffle=True,         # shuffles every epoch
    num_workers=0,
    drop_last=False
)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

# Peek at one batch
X_batch, y_batch = next(iter(train_loader))
print("Batch X shape:", X_batch.shape)   # [32, 1]
print("Batch y shape:", y_batch.shape)   # [32, 1]


In [ ]:
# -------------------------------------------------------
# 4. MODEL DEFINITION — nn.Module  vs  nn.Sequential
# -------------------------------------------------------

# Option A: nn.Sequential (quick, linear stacks)
model_seq = nn.Sequential(
    nn.Linear(1, 64),
    nn.ReLU(),
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Linear(32, 1)
)

# Option B: nn.Module (flexible, recommended)
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, x):
        return self.net(x)

model = MLP(input_dim=1, hidden_dim=64, output_dim=1)
print(model)
print("\nTotal params:", sum(p.numel() for p in model.parameters()))
print("Trainable params:", sum(p.numel() for p in model.parameters() if p.requires_grad))


In [ ]:
# -------------------------------------------------------
# 5. LOSS FUNCTIONS & OPTIMIZERS
# -------------------------------------------------------

# Common losses
mse_loss  = nn.MSELoss()          # regression
mae_loss  = nn.L1Loss()           # regression (robust)
ce_loss   = nn.CrossEntropyLoss() # multi-class classification (logits in, no softmax needed)
bce_loss  = nn.BCEWithLogitsLoss()# binary classification

# Optimizers
optimizer_sgd  = optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=1e-4)
optimizer_adam = optim.Adam(model.parameters(), lr=1e-3, betas=(0.9, 0.999))
optimizer_adamw= optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)

# We'll use Adam for training
optimizer = optimizer_adam

# Learning rate schedulers
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
# other options: CosineAnnealingLR, ReduceLROnPlateau

print("Loss functions and optimizers ready.")


In [ ]:
# -------------------------------------------------------
# 6. FULL TRAINING LOOP
# -------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model  = MLP(1, 64, 1).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

num_epochs = 30
train_losses, val_losses = [], []

for epoch in range(num_epochs):
    # ---- Train ----
    model.train()           # enables dropout / batchnorm training behaviour
    running_loss = 0.
    for X_b, y_b in train_loader:
        X_b, y_b = X_b.to(device), y_b.to(device)

        optimizer.zero_grad()       # 1. clear old gradients
        preds = model(X_b)          # 2. forward pass
        loss  = criterion(preds, y_b) # 3. compute loss
        loss.backward()             # 4. backprop
        optimizer.step()            # 5. update weights
        running_loss += loss.item()

    train_losses.append(running_loss / len(train_loader))

    # ---- Validate ----
    model.eval()            # disables dropout / batchnorm updates
    val_loss = 0.
    with torch.no_grad():   # no gradient computation needed
        for X_b, y_b in val_loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            preds    = model(X_b)
            val_loss += criterion(preds, y_b).item()
    val_losses.append(val_loss / len(val_loader))

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:3d}/{num_epochs} | "
              f"Train Loss: {train_losses[-1]:.4f} | "
              f"Val Loss:   {val_losses[-1]:.4f}")

print("\nTraining complete.")


In [ ]:
# -------------------------------------------------------
# 7. SAVING & LOADING CHECKPOINTS
# -------------------------------------------------------

# Save — recommended: state_dict only (portable)
torch.save(model.state_dict(), "model_weights.pth")

# Load
model_loaded = MLP(1, 64, 1)
model_loaded.load_state_dict(torch.load("model_weights.pth", map_location="cpu"))
model_loaded.eval()
print("Model loaded successfully.")

# Save full checkpoint (weights + optimizer state + epoch)
checkpoint = {
    "epoch": num_epochs,
    "model_state": model.state_dict(),
    "optimizer_state": optimizer.state_dict(),
    "train_losses": train_losses,
}
torch.save(checkpoint, "checkpoint.pth")

# Resume from checkpoint
ckpt = torch.load("checkpoint.pth", map_location="cpu")
model.load_state_dict(ckpt["model_state"])
optimizer.load_state_dict(ckpt["optimizer_state"])
start_epoch = ckpt["epoch"]
print(f"Resumed from epoch {start_epoch}")


In [ ]:
# -------------------------------------------------------
# 8. DEVICE MANAGEMENT (CPU / GPU)
# -------------------------------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Move tensors
t = torch.randn(3, 3).to(device)

# Move model
model = MLP(1, 64, 1).to(device)

# Check parameter device
for name, param in model.named_parameters():
    print(f"{name:20s} | device: {param.device}")
    break   # just show first one


In [ ]:
# -------------------------------------------------------
# 9. USEFUL UTILITIES
# -------------------------------------------------------

# --- Tensor ops recap ---
x = torch.randn(4, 3)
print("Mean:", x.mean())
print("Std: ", x.std())
print("Max: ", x.max())
print("Argmax:", x.argmax(dim=1))    # per-row argmax

# Reshape / view / squeeze / unsqueeze
flat = x.view(12)             # flatten to 1-D
col  = flat.unsqueeze(1)      # [12] -> [12, 1]
back = col.squeeze(1)         # [12, 1] -> [12]

# Stacking
a = torch.tensor([1., 2., 3.])
b = torch.tensor([4., 5., 6.])
stacked = torch.stack([a, b], dim=0)   # [2, 3]
catted  = torch.cat([a, b], dim=0)     # [6]
print("\nStacked shape:", stacked.shape)
print("Cat shape:    ", catted.shape)

# Gradient clipping (prevents exploding gradients)
# Typically used just before optimizer.step()
# torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
print("\nAll utilities demonstrated.")
